In [1]:
import truststore
truststore.inject_into_ssl()  # use Windows cert store so httpx/requests trust corporate root CAs
print("Ok")

Ok


In [2]:
%pwd

'e:\\Medicalchatbot\\research'

In [3]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [4]:
# Extract Data From the PDF File

def load_pdf_file(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )

    documents = loader.load()
    return documents


In [5]:
extracted_data = load_pdf_file(data='../Data/')


In [6]:
#extracted_data

In [7]:
# Split the Data into Text Chunks

def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20
    )

    text_chunks = text_splitter.split_documents(extracted_data)
    return text_chunks


In [8]:
text_chunks = text_split(extracted_data)
print("Length of Text Chunks", len(text_chunks))


Length of Text Chunks 2343


In [9]:
text_chunks


[Document(metadata={'producer': 'cairo 1.18.0 (https://cairographics.org)', 'creator': 'Mozilla Firefox 150.0.3', 'creationdate': '2026-05-19T22:53:10-05:00', 'source': '..\\Data\\Medicine.pdf', 'total_pages': 607, 'page': 1, 'page_label': '2'}, page_content='a LANGE medical book\nCURRENT\nESSENTIALS\nof MEDICINE\nF o u r t h E d i t i o n\nEdited by\nLawrence M. Tierney, Jr., MD\nProfessor of Medicine\nUniversity of California, San Francisco\nAssociate Chief of Medical Services\nVeterans Affairs Medical Center\nSan Francisco, California\nSanjay Saint, MD, MPH\nAssociate Chief of Medicine, Ann Arbor VA Medical Center\nDirector, VA/UM Patient Safety Enhancement Program\nProfessor of Internal Medicine, University of Michigan Medical School \nAnn Arbor, Michigan'),
 Document(metadata={'producer': 'cairo 1.18.0 (https://cairographics.org)', 'creator': 'Mozilla Firefox 150.0.3', 'creationdate': '2026-05-19T22:53:10-05:00', 'source': '..\\Data\\Medicine.pdf', 'total_pages': 607, 'page': 1, '

In [10]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"  # use cached model only; avoids httpx client-closed bug in huggingface_hub 1.x
from langchain_huggingface import HuggingFaceEmbeddings

In [11]:
# Download the Embeddings from Hugging Face

def download_hugging_face_embeddings():
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={"local_files_only": True},
    )
    return embeddings


In [12]:
embeddings = download_hugging_face_embeddings()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [13]:
query_result = embeddings.embed_query("Hello World")
print("Length", len(query_result))

Length 384


In [14]:
query_result

[-0.03447728976607323,
 0.03102319873869419,
 0.0067349751479923725,
 0.026109017431735992,
 -0.03936200216412544,
 -0.16030250489711761,
 0.06692400574684143,
 -0.006441437639296055,
 -0.047450508922338486,
 0.014758866280317307,
 0.07087535411119461,
 0.05552755296230316,
 0.019193312153220177,
 -0.02625133842229843,
 -0.010109523311257362,
 -0.026940515264868736,
 0.022307416424155235,
 -0.022226650267839432,
 -0.14969263970851898,
 -0.017493100836873055,
 0.007676243782043457,
 0.05435231328010559,
 0.003254458773881197,
 0.031725961714982986,
 -0.08462141454219818,
 -0.029405998066067696,
 0.05159568414092064,
 0.048124074935913086,
 -0.0033148014917969704,
 -0.05827920511364937,
 0.04196930304169655,
 0.022210700437426567,
 0.1281888335943222,
 -0.02233896777033806,
 -0.011656248942017555,
 0.06292837858200073,
 -0.03287629783153534,
 -0.09122606366872787,
 -0.031175322830677032,
 0.05269954353570938,
 0.04703483730554581,
 -0.08420302718877792,
 -0.030056210234761238,
 -0.020744

**Pinecone API key** is loaded from `.env` (see `PINECONE_API_KEY`). Do not paste it in the notebook.

In [15]:
from dotenv import load_dotenv
load_dotenv()

True

In [16]:
import os

In [17]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [18]:
from pinecone import Pinecone, ServerlessSpec
import os

pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "medicalchatbotindex"

existing = [i["name"] for i in pc.list_indexes()]
print(f"Existing indexes ({len(existing)}): {existing}")

if index_name in existing:
    print(f"Index already exists, reusing: {index_name}")
else:
    if len(existing) >= 5:
        # Free tier caps at 5 indexes. Auto-delete the first existing one that
        # isn't this notebook's target, to make room. Comment this out if you
        # want manual control.
        to_delete = existing[0]
        print(f"At index quota (5/5). Deleting {to_delete!r} to make room...")
        pc.delete_index(to_delete)
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    print(f"Created index: {index_name}")


Existing indexes (5): ['pin-raj447', 'medical', 'medicalchatbotindex', 'apidemo2', 'pinecone-demo']
Index already exists, reusing: medicalchatbotindex


In [19]:
import os
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [20]:
import os
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY

# Embed each chunk and upsert the embeddings into your Pinecone index.
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    index_name="medicalchatbotindex",
    embedding=embeddings,
)


In [21]:
# Load Existing index

from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_existing_index(
    index_name="medicalchatbotindex",
    embedding=embeddings
)


In [22]:
docsearch

In [23]:
retriever = docsearch.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)


In [24]:
retriever_docs = retriever.invoke("Tell me about mumps?")
retriever_docs

[Document(id='2e698579-dead-49ca-b106-caedff4cb4dd', metadata={'creationdate': '2026-05-19T22:53:10-05:00', 'creator': 'Mozilla Firefox 150.0.3', 'page': 290.0, 'page_label': '291', 'producer': 'cairo 1.18.0 (https://cairographics.org)', 'source': '..\\Data\\Medicine.pdf', 'total_pages': 607.0}, page_content='278 Current Essentials of Medicine\n8\nMumps (Epidemic Parotitis)\n■ Essentials of Diagnosis\n• Incubation period 14–24 days\n• Painful, swollen salivary glands, usually parotid; may be unilat-\neral; systemic symptoms of infection\n• Orchitis or oophoritis, meningoencephalitis, or pancreatitis may\noccur\n• Cerebrospinal \ue05auid shows lymphocytic pleocytosis in menin-\ngoencephalitis with hypoglycorrhachia\n• Diagnosis con\ue059rmed by isolation of virus in saliva or appearance\nof antibodies after second week'),
 Document(id='464b5375-326f-4695-a936-15c1f2699ea5', metadata={'creationdate': '2026-05-19T22:53:10-05:00', 'creator': 'Mozilla Firefox 150.0.3', 'page': 290.0, 'page_

In [25]:
from langchain_openai import OpenAI

llm = OpenAI(
    temperature=0.4,
    max_tokens=500
)


In [26]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)


In [27]:
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)


In [28]:
response = rag_chain.invoke({"input": "What is mumps ?"})
print(response["answer"])




Mumps is a viral infection that primarily affects the salivary glands, causing painful swelling. It has an incubation period of 14-24 days and can also lead to other complications such as orchitis, oophoritis, meningoencephalitis, or pancreatitis. Diagnosis is confirmed by isolating the virus in saliva or detecting antibodies after the second week of infection.
